# Step 1 - Ingest: CSV -> Parquet with context features, and the train/val/test split

Reads the 78 processed CSVs straight from the zip on Drive, turns every field into a number, adds context features computed from the packets *before* each packet (rates, inter-arrival times, new destination ports), and writes Parquet parts to `MyDrive/gotham_ids/data/ingest-<settings hash>/`. When all files are done it computes the time-blocked split and writes `reports/ingest/INGEST.md`.

* **Runtime:** CPU is enough (*Runtime -> Change runtime type -> CPU*).
* **Time:** roughly 30-60 min for the whole dataset; Drive space used: about 1 GB.
* **Disconnected?** Reconnect and run cells 2 and 4 again. Finished files are skipped and a file that was half done continues from its last checkpoint (every 128 MB of CSV), so at most a few minutes are repeated.
* **Results:** `MyDrive/gotham_ids/reports/ingest/INGEST.md` - copy what cell 6 prints into the chat.


In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 1 min; includes a crash + resume of ingest)
!python -m gids selftest


In [ ]:
#@title 4. Ingest every CSV, then compute the split (resumable: after a disconnect run cells 2 and 4 again)
!python -u -m gids ingest


In [ ]:
#@title 5. Report
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/ingest/INGEST.md").read()))


In [ ]:
#@title 6. Plain text of the report (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/ingest/INGEST.md").read())


### If something goes wrong
* `status` shows how many files are ingested and where a half-done file will continue.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* Changing an `ingest` setting writes to a new `ingest-<hash>` folder instead of mixing results; changing a `split` setting only recomputes the split (seconds), not the ingest.
* To try other split settings without editing the repository: `!python -m gids --set split.block_h=2 ingest`.


In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30
